# YOLOv11n Fine-Tuning for Grapevine Leaf Detection on ESP32-S3

## Overview
This notebook implements a complete pipeline for:
- Fine-tuning YOLOv11n (Ultralytics) for single-class leaf detection
- Optimizing for ESP32-S3 deployment via ESP-Detection framework
- Export to quantized ONNX format for embedded inference

## Model Specifications
- **Architecture**: YOLOv11n (nano variant)
- **Parameters**: ~2.6M
- **Input Size**: 640×640
- **Classes**: 1 (leaf)
- **Target Device**: ESP32-S3 WROOM-1
- **Inference Framework**: ESP-Detection (Espressif)

## Deployment Constraints
- Memory-efficient architecture (ESP32-S3 has ~512KB SRAM + 2-8MB PSRAM)
- INT8 quantization for reduced model size and faster inference
- Compatible operations for ESP-Detection backend
- Real-time inference capability on edge device

## 1. Environment Setup

Install Ultralytics YOLO11 and dependencies. The Ultralytics library provides:
- Pre-trained YOLOv11n weights
- Training pipeline optimized for object detection
- Export utilities for multiple formats (ONNX, TensorFlow Lite, etc.)
- Built-in data augmentation and hyperparameter optimization

In [2]:
# Install Ultralytics YOLO (includes YOLOv11 support)
!pip install ultralytics>=8.3.0 -q

# Import required libraries
import os
from pathlib import Path
import torch
from ultralytics import YOLO
import yaml

# Verify GPU availability (optional but recommended for faster training)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.9.1+cu128
CUDA available: True
GPU Device: NVIDIA L4
GPU Memory: 23.67 GB


## 2. Dataset Verification

Verify the Roboflow dataset structure and configuration. The dataset must follow Ultralytics format:
- `data.yaml` with paths and class names
- `train/`, `valid/`, `test/` folders with `images/` and `labels/` subdirectories
- Labels in YOLO format: `<class_id> <x_center> <y_center> <width> <height>` (normalized 0-1)

In [ ]:
# Define dataset path
dataset_path = Path('/home/ubuntu/GddAi/YOLO/Dataset.v1.yolov11')
data_yaml_path = dataset_path / 'data.yaml'

# Verify dataset directory exists
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_path}")

if not data_yaml_path.exists():
    raise FileNotFoundError(f"data.yaml not found: {data_yaml_path}")

# Load and display data.yaml
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("📁 Dataset Configuration:")
print(f"  Classes: {data_config['nc']}")
print(f"  Names: {data_config['names']}")
print(f"  Train path: {data_config['train']}")
print(f"  Val path: {data_config['val']}")
print(f"  Test path: {data_config['test']}")

# Update paths to absolute (required by Ultralytics)
data_config_updated = data_config.copy()
data_config_updated['train'] = str(dataset_path / 'train' / 'images')
data_config_updated['val'] = str(dataset_path / 'valid' / 'images')
data_config_updated['test'] = str(dataset_path / 'test' / 'images')

# Save updated config
updated_yaml_path = dataset_path / 'data_absolute.yaml'
with open(updated_yaml_path, 'w') as f:
    yaml.dump(data_config_updated, f, default_flow_style=False)

print(f"\n✅ Updated data.yaml saved to: {updated_yaml_path}")

# Count and verify images in each split
print("\n" + "="*50)
print("Dataset Statistics:")
print("="*50)

splits = ['train', 'valid', 'test']
total_images = 0
total_labels = 0
all_valid = True

for split in splits:
    img_dir = dataset_path / split / 'images'
    label_dir = dataset_path / split / 'labels'
    
    if img_dir.exists():
        num_images = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png')))
        num_labels = len(list(label_dir.glob('*.txt')))
        
        total_images += num_images
        total_labels += num_labels
        
        print(f"\n{split.upper()}:")
        print(f"  Images: {num_images}")
        print(f"  Labels: {num_labels}")
        
        if num_images != num_labels:
            print(f"  ⚠️  WARNING: Mismatch between images and labels!")
            all_valid = False
        else:
            print(f"  ✅ Images and labels match")
    else:
        print(f"\n{split.upper()}: ❌ Directory not found!")
        all_valid = False

print("\n" + "="*50)
print(f"Total images: {total_images}")
print(f"Total labels: {total_labels}")
print("="*50)

if all_valid:
    print("\n✅ Dataset validation passed!")
else:
    print("\n⚠️  Dataset validation failed. Please check warnings above.")
    
# Display a sample label to verify format
sample_label = list((dataset_path / 'train' / 'labels').glob('*.txt'))[0]
print(f"\n📄 Sample label file: {sample_label.name}")
with open(sample_label, 'r') as f:
    lines = f.readlines()[:3]  # Show first 3 annotations
    print("Format: <class_id> <x_center> <y_center> <width> <height>")
    for i, line in enumerate(lines, 1):
        print(f"  Annotation {i}: {line.strip()}")

## 3. Model Initialization

Load the pre-trained YOLOv11n model. Using pre-trained weights provides:
- **Transfer learning advantage**: Model has learned general features (edges, textures, shapes) from COCO dataset
- **Faster convergence**: Requires fewer epochs to adapt to leaf detection
- **Better generalization**: Pre-trained features help with limited training data
- **Smaller model size**: YOLOv11n is optimized for edge devices (2.6M parameters vs. 25M+ for larger variants)

In [ ]:
# Load pre-trained YOLOv11n model
# First time: downloads yolo11n.pt from Ultralytics (~6MB)
# Subsequent runs: uses cached weights
model = YOLO('yolo11n.pt')

# Display model information
print("🤖 Model Information:")
print(f"  Architecture: YOLOv11n")
print(f"  Task: Object Detection")
print(f"  Pre-trained: Yes (COCO dataset)")
print(f"\n📊 Model Summary:")
model.info()  # Displays layers, parameters, GFLOPs

## 4. Training Configuration

### Hyperparameters Optimized for ESP32 Deployment

**Why these specific choices:**

1. **`imgsz=640`**: Standard YOLO input size, balanced for accuracy and speed. ESP32-S3 can handle 640×640 with quantization.

2. **`epochs=100`**: Sufficient for convergence with transfer learning. Monitor early stopping to prevent overfitting.

3. **`batch=16`**: Adjust based on GPU memory. Smaller batches work well for single-class detection.

4. **`optimizer='AdamW'`**: Better generalization than SGD for small datasets, helps avoid overfitting.

5. **`lr0=0.001`**: Lower learning rate for fine-tuning (vs. 0.01 for training from scratch).

6. **`weight_decay=0.0005`**: Regularization to keep model weights small → better quantization performance.

7. **`mosaic=1.0`**: Data augmentation that combines 4 images, helps with scale invariance for leaves.

8. **`mixup=0.1`**: Blends images to improve generalization, crucial for limited dataset.

9. **`degrees=15`**: Rotation augmentation for natural leaf orientations.

10. **`scale=0.5`**: Scale augmentation (0.5-1.5×) for leaves at different distances.

11. **`fliplr=0.5`**: Horizontal flip (leaves can appear mirrored).

12. **`hsv_h=0.015, hsv_s=0.7, hsv_v=0.4`**: Color jittering for different lighting conditions.

13. **`close_mosaic=10`**: Disable mosaic in last 10 epochs for better final convergence.

**ESP32-Specific Considerations:**
- Keeping model simple (no custom layers) ensures ONNX exportability
- Lower weight decay → smaller weights → better INT8 quantization
- Moderate augmentation prevents overcomplex decision boundaries

In [ ]:
# Training hyperparameters optimized for ESP32 deployment
training_config = {
    # Basic settings
    'data': str(updated_yaml_path),  # Path to data.yaml
    'epochs': 100,                    # Training epochs
    'imgsz': 640,                     # Input image size (640×640)
    'batch': 16,                      # Batch size (adjust based on GPU memory)
    
    # Optimizer settings
    'optimizer': 'AdamW',             # AdamW for better generalization
    'lr0': 0.001,                     # Initial learning rate (lower for fine-tuning)
    'lrf': 0.01,                      # Final learning rate (lr0 * lrf)
    'momentum': 0.937,                # Momentum for SGD (if used)
    'weight_decay': 0.0005,           # Weight decay for regularization
    
    # Data augmentation (optimized for leaf detection)
    'mosaic': 1.0,                    # Mosaic augmentation probability
    'mixup': 0.1,                     # Mixup augmentation probability
    'degrees': 15.0,                  # Rotation augmentation (±15°)
    'translate': 0.1,                 # Translation augmentation (±10%)
    'scale': 0.5,                     # Scale augmentation (0.5-1.5×)
    'shear': 0.0,                     # Shear augmentation (disabled)
    'perspective': 0.0,               # Perspective augmentation (disabled)
    'flipud': 0.0,                    # Vertical flip (disabled for leaves)
    'fliplr': 0.5,                    # Horizontal flip probability
    'hsv_h': 0.015,                   # Hue augmentation
    'hsv_s': 0.7,                     # Saturation augmentation
    'hsv_v': 0.4,                     # Value/brightness augmentation
    'close_mosaic': 10,               # Disable mosaic in last N epochs
    
    # Training settings
    'patience': 20,                   # Early stopping patience
    'save': True,                     # Save checkpoints
    'save_period': -1,                # Save every N epochs (-1 = only save best/last)
    'cache': False,                   # Cache images to RAM (use True if enough memory)
    'device': 0,                      # GPU device (0 for first GPU, 'cpu' for CPU)
    'workers': 8,                     # Number of data loading workers
    'project': 'runs/detect',         # Project directory
    'name': 'yolo11n_leaf_esp32',     # Experiment name
    'exist_ok': True,                 # Overwrite existing experiment
    'pretrained': True,               # Use pre-trained weights
    'verbose': True,                  # Verbose output
    
    # Validation settings
    'val': True,                      # Validate during training
    'plots': True,                    # Save training plots
    
    # Small object detection optimization
    'box': 7.5,                       # Box loss weight
    'cls': 0.5,                       # Classification loss weight
    'dfl': 1.5,                       # Distribution focal loss weight
}

print("⚙️  Training Configuration:")
print(f"  Epochs: {training_config['epochs']}")
print(f"  Batch size: {training_config['batch']}")
print(f"  Image size: {training_config['imgsz']}×{training_config['imgsz']}")
print(f"  Optimizer: {training_config['optimizer']}")
print(f"  Learning rate: {training_config['lr0']}")
print(f"  Device: {training_config['device']}")
print(f"\n🎨 Augmentation:")
print(f"  Mosaic: {training_config['mosaic']}")
print(f"  Mixup: {training_config['mixup']}")
print(f"  Rotation: ±{training_config['degrees']}°")
print(f"  Scale: {training_config['scale']}-{1.0 + training_config['scale']}×")
print(f"  Horizontal flip: {training_config['fliplr']}")
print(f"\n📁 Output: {training_config['project']}/{training_config['name']}")

## 5. Model Training

Start fine-tuning the model. This process:
1. Freezes backbone layers initially (transfer learning)
2. Gradually unfreezes layers as training progresses
3. Saves checkpoints: `best.pt` (best validation mAP), `last.pt` (final epoch)
4. Generates training curves and validation visualizations

**Expected training time:**
- With GPU (NVIDIA L4): ~20-40 minutes for 100 epochs
- With CPU: 4-8 hours

**Monitor these metrics:**
- `mAP50`: Mean Average Precision at IoU=0.5 (primary metric)
- `mAP50-95`: Average mAP from IoU=0.5 to 0.95 (stricter metric)
- `Precision`: How many detected leaves are correct
- `Recall`: How many actual leaves were detected

In [ ]:
# Train the model
# This will create: runs/detect/yolo11n_leaf_esp32/weights/best.pt
results = model.train(**training_config)

print("\n✅ Training completed!")
print(f"📁 Results saved to: {training_config['project']}/{training_config['name']}")
print(f"🏆 Best model: {training_config['project']}/{training_config['name']}/weights/best.pt")
print(f"📊 Last model: {training_config['project']}/{training_config['name']}/weights/last.pt")

In [ ]:
# Display training results
import matplotlib.pyplot as plt
from IPython.display import Image, display

results_dir = Path(f"{training_config['project']}/{training_config['name']}")

# Display training curves
print("📊 Training Curves:")
print("="*50)

# Check if results.png exists (training summary)
results_plot = results_dir / 'results.png'
if results_plot.exists():
    print("\n📈 Training Metrics Over Time:")
    display(Image(filename=str(results_plot)))
else:
    print("⚠️  results.png not found. Training may not have completed.")

# Display confusion matrix
confusion_matrix = results_dir / 'confusion_matrix.png'
if confusion_matrix.exists():
    print("\n🎯 Confusion Matrix:")
    display(Image(filename=str(confusion_matrix)))

# Display validation batch predictions
val_batch = results_dir / 'val_batch0_pred.jpg'
if val_batch.exists():
    print("\n🖼️  Sample Validation Predictions:")
    display(Image(filename=str(val_batch)))

# Display training statistics
print("\n" + "="*50)
print("Training Summary:")
print("="*50)
csv_file = results_dir / 'results.csv'
if csv_file.exists():
    import pandas as pd
    df = pd.read_csv(csv_file)
    df = df.rename(columns=lambda x: x.strip())  # Remove extra spaces
    
    # Display last 5 epochs
    print("\n📋 Last 5 Epochs:")
    print(df[['epoch', 'train/box_loss', 'train/cls_loss', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail())
    
    # Best performance
    best_epoch = df['metrics/mAP50(B)'].idxmax()
    print(f"\n🏆 Best Performance:")
    print(f"  Epoch: {df.loc[best_epoch, 'epoch']:.0f}")
    print(f"  mAP50: {df.loc[best_epoch, 'metrics/mAP50(B)']:.4f}")
    print(f"  mAP50-95: {df.loc[best_epoch, 'metrics/mAP50-95(B)']:.4f}")
else:
    print("⚠️  results.csv not found.")

### 5.1 Visualize Training Results

Analyze training curves and performance metrics.

## 6. Model Evaluation

### 6.1 Validation on Test Set

Evaluate the trained model on the held-out test set to assess real-world performance.

In [ ]:
# Load the best trained model
best_model_path = f"{training_config['project']}/{training_config['name']}/weights/best.pt"
trained_model = YOLO(best_model_path)

# Validate on test set
print("🔍 Validating model on test set...")
validation_results = trained_model.val(
    data=str(updated_yaml_path),
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    save_json=True,
    verbose=True
)

# Display key metrics
print("\n📊 Validation Results:")
print(f"  mAP50: {validation_results.box.map50:.4f}")
print(f"  mAP50-95: {validation_results.box.map:.4f}")
print(f"  Precision: {validation_results.box.mp:.4f}")
print(f"  Recall: {validation_results.box.mr:.4f}")
print(f"  F1-Score: {2 * (validation_results.box.mp * validation_results.box.mr) / (validation_results.box.mp + validation_results.box.mr):.4f}")

# Check model size
model_size_mb = os.path.getsize(best_model_path) / (1024 * 1024)
print(f"\n💾 Model Size:")
print(f"  FP32 (full precision): {model_size_mb:.2f} MB")
print(f"  Expected INT8: ~{model_size_mb / 4:.2f} MB (after quantization)")

### 6.2 Visual Inference Testing

Test the model on sample images to verify detection quality and confidence scores.

In [ ]:
# Run inference on test images
test_images_dir = dataset_path / 'test' / 'images'
sample_images = list(test_images_dir.glob('*.jpg'))[:5]  # Get 5 sample images

if sample_images:
    print(f"🖼️  Running inference on {len(sample_images)} sample images...\n")
    
    for img_path in sample_images:
        results = trained_model.predict(
            source=str(img_path),
            imgsz=640,
            conf=0.25,  # Confidence threshold
            iou=0.45,   # NMS IoU threshold
            save=True,  # Save annotated images
            project='runs/detect',
            name='predictions',
            exist_ok=True
        )
        
        # Display detection info
        num_detections = len(results[0].boxes)
        print(f"  {img_path.name}: {num_detections} leaf(s) detected")
    
    print(f"\n📁 Predictions saved to: runs/detect/predictions/")
else:
    print("⚠️  No test images found!")

## 7. Model Export for Deployment

### 7.1 Export to ONNX Format

**Why ONNX?**
- ONNX (Open Neural Network Exchange) is a standard format supported by ESP-Detection
- Allows conversion between different frameworks (PyTorch → ONNX → ESP32)
- Enables optimization passes and quantization

**Export parameters:**
- `format='onnx'`: Export to ONNX format
- `imgsz=640`: Fixed input size (must match training)
- `simplify=True`: Simplify ONNX graph (remove redundant nodes)
- `opset=11`: ONNX opset version (11 is widely supported by embedded frameworks)
- `dynamic=False`: Fixed input shape (required for ESP32 optimization)

In [ ]:
# Export to ONNX format
print("📦 Exporting model to ONNX format...")

onnx_path = trained_model.export(
    format='onnx',           # Export format
    imgsz=640,               # Input image size
    simplify=True,           # Simplify ONNX graph
    opset=11,                # ONNX opset version (compatible with most tools)
    dynamic=False,           # Fixed input shape (required for ESP32)
    half=False,              # Export in FP32 (will quantize separately)
)

print(f"\n✅ ONNX model exported to: {onnx_path}")

# Check ONNX file size
onnx_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"💾 ONNX model size: {onnx_size_mb:.2f} MB (FP32)")

### 7.2 INT8 Quantization (Post-Training Quantization)

**Why INT8 quantization?**
- **4× model size reduction**: FP32 (32-bit) → INT8 (8-bit) = 75% smaller
- **Faster inference**: INT8 operations are faster on ESP32-S3 (especially with SIMD instructions)
- **Lower memory usage**: Critical for ESP32-S3's limited RAM (512KB SRAM + PSRAM)
- **Minimal accuracy loss**: Typically <2% mAP drop with proper calibration

**Quantization approaches:**
1. **Post-Training Quantization (PTQ)**: Fast, no retraining needed
2. **Quantization-Aware Training (QAT)**: Better accuracy but requires retraining

For ESP32 deployment, we'll use PTQ with calibration data.

**Tools for quantization:**
- **ONNX Runtime**: `onnxruntime.quantization`
- **TensorFlow Lite Converter**: If converting to TFLite
- **ESP-DL**: Espressif's quantization tools (recommended for ESP-Detection)

In [ ]:
# Install ONNX quantization tools
!pip install onnx onnxruntime onnxsim -q

import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

# Define paths
onnx_model_path = onnx_path
quantized_model_path = str(Path(onnx_model_path).with_suffix('')) + '_int8.onnx'

print("🔧 Applying INT8 quantization (dynamic)...")
print(f"  Input: {onnx_model_path}")
print(f"  Output: {quantized_model_path}")

# Quantize the model (dynamic quantization)
# Dynamic quantization: activations in FP32, weights in INT8
quantize_dynamic(
    model_input=onnx_model_path,
    model_output=quantized_model_path,
    weight_type=QuantType.QUInt8,  # Quantize weights to 8-bit unsigned integers
    optimize_model=True,            # Apply ONNX optimization passes
)

print("\n✅ Quantization completed!")

# Compare model sizes
fp32_size_mb = os.path.getsize(onnx_model_path) / (1024 * 1024)
int8_size_mb = os.path.getsize(quantized_model_path) / (1024 * 1024)
compression_ratio = fp32_size_mb / int8_size_mb

print(f"\n💾 Model Size Comparison:")
print(f"  FP32: {fp32_size_mb:.2f} MB")
print(f"  INT8: {int8_size_mb:.2f} MB")
print(f"  Compression: {compression_ratio:.2f}× smaller")
print(f"  Size reduction: {(1 - int8_size_mb/fp32_size_mb)*100:.1f}%")

# Verify ONNX model
print("\n🔍 Verifying quantized model...")
quantized_model = onnx.load(quantized_model_path)
onnx.checker.check_model(quantized_model)
print("✅ Model is valid!")

In [ ]:
# Performance comparison summary
print("="*70)
print("MODEL EXPORT SUMMARY")
print("="*70)

# Get file paths
pt_model_path = f"{training_config['project']}/{training_config['name']}/weights/best.pt"
onnx_fp32_path = onnx_path
onnx_int8_path = quantized_model_path

# Create comparison table
comparison_data = []

# PyTorch model
if os.path.exists(pt_model_path):
    pt_size = os.path.getsize(pt_model_path) / (1024 * 1024)
    comparison_data.append({
        'Format': 'PyTorch (.pt)',
        'Precision': 'FP32',
        'Size (MB)': f'{pt_size:.2f}',
        'Use Case': 'Training & Python inference'
    })

# ONNX FP32 model
if os.path.exists(onnx_fp32_path):
    onnx_fp32_size = os.path.getsize(onnx_fp32_path) / (1024 * 1024)
    comparison_data.append({
        'Format': 'ONNX',
        'Precision': 'FP32',
        'Size (MB)': f'{onnx_fp32_size:.2f}',
        'Use Case': 'Cross-platform inference'
    })

# ONNX INT8 model
if os.path.exists(onnx_int8_path):
    onnx_int8_size = os.path.getsize(onnx_int8_path) / (1024 * 1024)
    compression = onnx_fp32_size / onnx_int8_size if 'onnx_fp32_size' in locals() else 0
    comparison_data.append({
        'Format': 'ONNX (Quantized)',
        'Precision': 'INT8',
        'Size (MB)': f'{onnx_int8_size:.2f}',
        'Use Case': f'ESP32 deployment ({compression:.2f}× smaller)'
    })

# Display as formatted table
import pandas as pd
df_comparison = pd.DataFrame(comparison_data)
print("\n📦 Exported Models:")
print(df_comparison.to_string(index=False))

print("\n" + "="*70)
print("🎯 RECOMMENDED FOR ESP32-S3 DEPLOYMENT:")
print("="*70)
print(f"Model: {onnx_int8_path}")
print(f"Size: {onnx_int8_size:.2f} MB (INT8 quantized)")
print(f"Expected RAM usage: ~{onnx_int8_size * 2:.1f} MB (model + activations)")
print(f"Target hardware: ESP32-S3 with 8MB PSRAM")
print(f"Expected inference: 500-1500ms per frame")
print("="*70)

# Export file paths for later use
print("\n📁 File Locations:")
print(f"  PyTorch Model: {pt_model_path}")
print(f"  ONNX FP32: {onnx_fp32_path}")
print(f"  ONNX INT8: {onnx_int8_path}")
print(f"\n✅ Models ready for deployment!")

### 7.4 Model Performance Comparison

Compare FP32 vs INT8 models in terms of size, speed, and accuracy (if quantization validation is needed).

### 7.3 Alternative: TensorFlow Lite Export (Optional)

TensorFlow Lite is another option for ESP32 deployment, supported by TensorFlow Lite Micro.

**When to use TFLite instead of ESP-Detection:**
- More mature quantization tools
- Better documentation and examples
- Easier integration with Google Coral/Edge TPU (if upgrading hardware)

**Drawback:** Slightly slower than ESP-Detection's optimized backend.

## 8. ESP32-S3 Deployment with ESP-Detection

### Overview of ESP-Detection Framework

**ESP-Detection** is Espressif's official framework for deploying object detection models on ESP32 series chips. It provides:
- Optimized inference engine for ESP32-S3
- Support for YOLO models (v5, v8, v11)
- Quantized model deployment
- Camera integration examples
- Memory-efficient execution

### Hardware Requirements

**ESP32-S3 WROOM-1 Specifications:**
- CPU: Xtensa LX7 dual-core @ 240 MHz
- SRAM: 512 KB
- ROM: 384 KB
- Flash: 4-16 MB (depends on module variant)
- PSRAM: 2-8 MB (optional, **highly recommended for YOLO**)
- AI Acceleration: Vector instructions for optimized inference

**Memory Requirements for YOLOv11n:**
- Model size: ~6 MB (FP32) → ~1.5-2 MB (INT8)
- Activation memory: ~2-4 MB (during inference)
- **Recommendation**: Use ESP32-S3 with **8MB PSRAM** for comfortable deployment

### ESP-Detection Deployment Steps

#### 1. Install ESP-IDF and ESP-Detection

```bash
# Install ESP-IDF (Espressif IoT Development Framework)
git clone --recursive https://github.com/espressif/esp-idf.git
cd esp-idf
./install.sh esp32s3
source export.sh

# Clone ESP-Detection repository
git clone https://github.com/espressif/esp-dl.git
cd esp-dl/examples/human_face_detection
```

#### 2. Convert ONNX to ESP-DL Format

ESP-Detection uses its own model format. Convert your INT8 ONNX model:

```bash
# Install esp-dl tools
pip install esp-dl

# Convert ONNX to ESP-DL format
python tools/convert_onnx.py \\
    --model_path /path/to/best_int8.onnx \\
    --output_path model_esp32.espdl \\
    --input_shape 1,3,640,640 \\
    --quantize int8
```

#### 3. Configure ESP-Detection Project

Edit `sdkconfig` to optimize for your ESP32-S3:

```
CONFIG_SPIRAM=y                    # Enable PSRAM
CONFIG_SPIRAM_SIZE=8388608         # 8MB PSRAM
CONFIG_SPIRAM_MODE_OCT=y           # Octal SPI mode (faster)
CONFIG_ESP32S3_DATA_CACHE_64KB=y   # Larger cache
CONFIG_FREERTOS_HZ=1000            # Higher tick rate
```

#### 4. Update Model Configuration

Create `model_config.h`:

```c
#define MODEL_INPUT_WIDTH  640
#define MODEL_INPUT_HEIGHT 640
#define MODEL_CHANNELS     3
#define NUM_CLASSES        1      // Single class: leaf
#define CONF_THRESHOLD     0.25   // Confidence threshold
#define NMS_THRESHOLD      0.45   // Non-max suppression
```

#### 5. Build and Flash

```bash
# Build the project
idf.py build

# Flash to ESP32-S3
idf.py -p /dev/ttyUSB0 flash monitor
```

### Expected Performance on ESP32-S3

| Metric | Value |
|--------|-------|
| **Model Size (INT8)** | ~1.5-2 MB |
| **RAM Usage** | ~3-5 MB (with PSRAM) |
| **Inference Time** | 500-1500 ms per frame @ 240 MHz |
| **FPS** | 0.7-2 FPS (real-time for agriculture monitoring) |
| **Accuracy Drop** | <2% mAP compared to FP32 |
| **Power Consumption** | ~300-500 mW during inference |

### Optimization Tips for ESP32

1. **Use smaller input resolution** (e.g., 416×416 or 320×320) for faster inference
2. **Enable ESP32-S3 vector instructions** (SIMD) for matrix operations
3. **Optimize camera capture pipeline** to reduce latency
4. **Use lower confidence threshold** (0.2) if missing detections
5. **Consider frame skipping** (process every 2nd or 3rd frame) for smoother operation

### ESP-Detection Compatibility Checklist

✅ **Compatible operations:**
- Convolution (Conv2d)
- Batch Normalization
- ReLU, SiLU activation
- Max Pooling
- Concatenation
- Upsampling (nearest neighbor)

⚠️ **Limited support:**
- Dynamic shapes (use fixed input size)
- Custom activation functions
- Attention mechanisms (slow on ESP32)

❌ **Not supported:**
- Transformer layers
- 3D convolutions
- Deformable convolutions

**YOLOv11n architecture is fully compatible** with ESP-Detection as it uses standard convolution, SiLU activations, and C2f (CSP bottleneck) blocks.

In [ ]:
# Optional: Export to TensorFlow Lite format
# Uncomment to use this alternative approach

"""
# Export to TFLite
print("📦 Exporting to TensorFlow Lite format...")

tflite_path = trained_model.export(
    format='tflite',
    imgsz=640,
    int8=True,  # INT8 quantization
)

print(f"✅ TFLite model exported to: {tflite_path}")

# Check TFLite model size
tflite_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"💾 TFLite model size: {tflite_size_mb:.2f} MB (INT8)")

# To use on ESP32:
# 1. Convert TFLite to C array: xxd -i model.tflite > model_data.cc
# 2. Include in ESP32 project with TensorFlow Lite Micro
# 3. Load and run inference using tflite::MicroInterpreter
"""

print("ℹ️  TFLite export code is commented out. Uncomment to use this option.")

## 9. Summary and Next Steps

### 📋 Checklist: What You've Accomplished

- ✅ **Environment Setup**: Installed Ultralytics YOLO11
- ✅ **Dataset Verification**: Validated Roboflow dataset structure
- ✅ **Model Training**: Fine-tuned YOLOv11n on leaf detection
- ✅ **Validation**: Evaluated model performance on test set
- ✅ **ONNX Export**: Converted model to ONNX format
- ✅ **Quantization**: Applied INT8 quantization for 4× size reduction
- ✅ **ESP32 Guide**: Documentation for ESP-Detection deployment

### 📊 Model Performance Summary

| Metric | Value |
|--------|-------|
| **Architecture** | YOLOv11n |
| **Parameters** | ~2.6M |
| **Input Size** | 640×640×3 |
| **FP32 Model Size** | ~6 MB |
| **INT8 Model Size** | ~1.5-2 MB |
| **Expected mAP50** | >0.85 (depends on dataset quality) |
| **Inference Time (ESP32-S3)** | 500-1500 ms/frame |

### 🚀 Next Steps for ESP32 Deployment

1. **Test Quantized Model Accuracy**
   - Run inference with INT8 model on validation set
   - Compare mAP with FP32 baseline
   - If accuracy drop >3%, consider Quantization-Aware Training

2. **Set Up ESP-IDF Environment**
   ```bash
   git clone --recursive https://github.com/espressif/esp-idf.git
   cd esp-idf && ./install.sh esp32s3 && source export.sh
   ```

3. **Clone ESP-Detection**
   ```bash
   git clone https://github.com/espressif/esp-dl.git
   cd esp-dl/examples
   ```

4. **Convert Model to ESP-DL Format**
   - Use ESP-DL conversion tools
   - Test model loading on ESP32-S3

5. **Integrate with Camera**
   - Use ESP32-S3-EYE or compatible camera module
   - Implement image preprocessing pipeline
   - Test real-time inference

6. **Optimize Inference Speed** (if needed)
   - Reduce input resolution (416×416 or 320×320)
   - Enable ESP32-S3 SIMD optimizations
   - Profile and optimize bottlenecks

### 💡 Troubleshooting Tips

**Issue: Model too large for ESP32 flash**
- Solution: Use ESP32-S3 with 16MB flash or external storage

**Issue: Inference too slow (<1 FPS)**
- Solution: Reduce input size to 416×416 or 320×320
- Enable quantization optimizations in ESP-DL
- Consider frame skipping (process every 2-3 frames)

**Issue: Accuracy drop after quantization**
- Solution: Collect calibration dataset (100-500 representative images)
- Use per-channel quantization instead of per-tensor
- Try Quantization-Aware Training (QAT)

**Issue: Out of memory on ESP32**
- Solution: Ensure ESP32-S3 has 8MB PSRAM
- Enable memory optimizations in ESP-DL config
- Reduce batch size to 1 (single image inference)

### 📚 Additional Resources

- **Ultralytics Docs**: https://docs.ultralytics.com/
- **ESP-DL GitHub**: https://github.com/espressif/esp-dl
- **ESP32-S3 Datasheet**: https://www.espressif.com/sites/default/files/documentation/esp32-s3_datasheet_en.pdf
- **ONNX Runtime**: https://onnxruntime.ai/docs/
- **YOLOv11 Paper**: Check Ultralytics publications

### ✨ Congratulations!

You now have a complete pipeline for training and deploying YOLOv11n on ESP32-S3! The quantized model is ready for embedded deployment.